# 120 Years of Olympic History

Grasso Gianni & Ismael Trentin

## Dataset

This is a historical dataset on the modern Olympic Games, including all the Games from Athens 1896 to Rio 2016. The data was scraped from www.sports-reference.com in May 2018. More info can be found [here](https://www.kaggle.com/datasets/heesoo37/120-years-of-olympic-history-athletes-and-results).


## Code

In [21]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# pio.renderers.default = "notebook"
from plotly.subplots import make_subplots

In [22]:
df = pd.read_csv('data/athlete_events.csv')

In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 271116 entries, 0 to 271115
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   ID      271116 non-null  int64  
 1   Name    271116 non-null  str    
 2   Sex     271116 non-null  str    
 3   Age     261642 non-null  float64
 4   Height  210945 non-null  float64
 5   Weight  208241 non-null  float64
 6   Team    271116 non-null  str    
 7   NOC     271116 non-null  str    
 8   Games   271116 non-null  str    
 9   Year    271116 non-null  int64  
 10  Season  271116 non-null  str    
 11  City    271116 non-null  str    
 12  Sport   271116 non-null  str    
 13  Event   271116 non-null  str    
 14  Medal   39783 non-null   str    
dtypes: float64(3), int64(2), str(10)
memory usage: 31.0 MB


In [24]:
medals = df[df["Medal"].notna()]

# Totale per NOC
total = medals.groupby("NOC").size().reset_index(name="total")
top20_noc = total.nlargest(20, "total")["NOC"].tolist()

# Breakdown per tipo
breakdown = (medals[medals["NOC"].isin(top20_noc)]
             .groupby(["NOC", "Medal"])
             .size()
             .reset_index(name="count"))

# Ordine per totale crescente (così in alto c'è il maggiore nei grafici orizzontali)
order = (total[total["NOC"].isin(top20_noc)]
         .sort_values("total", ascending=True)["NOC"].tolist())

total_top = total[total["NOC"].isin(top20_noc)].set_index("NOC").reindex(order)

color_map = {"Gold": "#FFD700", "Silver": "#C0C0C0", "Bronze": "#CD7F32"}
medal_order = ["Bronze", "Silver", "Gold"]  # stacking order

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.38, 0.62],
    shared_yaxes=True,
    horizontal_spacing=0.04
)

# ── Pannello sinistro: barre totali (scala log) ───────────────────────────
fig.add_trace(
    go.Bar(
        y=order,
        x=total_top["total"],
        orientation="h",
        marker_color="#6272a4",
        showlegend=False,
        name="Totale",
        hovertemplate="%{y}: %{x} medaglie<extra></extra>"
    ),
    row=1, col=1
)

# ── Pannello destro: 100% stacked per tipo medaglia ───────────────────────
for medal in medal_order:
    sub = breakdown[breakdown["Medal"] == medal].set_index("NOC")
    # calcola percentuale
    total_dict = total.set_index("NOC")["total"]
    pct = [(sub.loc[noc, "count"] / total_dict[noc] * 100
            if noc in sub.index else 0)
           for noc in order]

    fig.add_trace(
        go.Bar(
            y=order,
            x=pct,
            orientation="h",
            name=medal,
            marker_color=color_map[medal],
            hovertemplate=f"{medal}: %{{x:.1f}}%<extra></extra>"
        ),
        row=1, col=2
    )

# ── Layout ────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(text="Medaglie Olimpiche Per Tipo Nelle Nazioni", font_size=26, x=0),
    barmode="stack",
    template="plotly_white",
    height=620,
    legend=dict(title="Tipo medaglia", traceorder="reversed",
                x=1.01, y=0.5, xanchor="left"),
    margin=dict(l=20, r=120, t=80, b=40),
    annotations=[
        dict(text="Totale medaglie", x=0.17, y=1.04,
             xref="paper", yref="paper", showarrow=False, font_size=13),
        dict(text="Quota per tipo [%]", x=0.72, y=1.04,
             xref="paper", yref="paper", showarrow=False, font_size=13),
    ]
)

fig.update_xaxes(type="log", title_text="Medaglie totali (log)", row=1, col=1)
fig.update_xaxes(title_text="Quota [%]", range=[0, 100], row=1, col=2)
fig.update_yaxes(title_text="Paese", row=1, col=1)

fig.show()

In [25]:
# ── Partecipazione atletica: totale per anno + quota M/F ──────────────────

part = (df.groupby(["Year", "Sex"])
          .size()
          .reset_index(name="count"))

total_year = part.groupby("Year")["count"].sum().reset_index(name="total")
years = total_year.sort_values("Year")["Year"].tolist()

color_sex = {"M": "#4C72B0", "F": "#DD8452"}
sex_order = ["M", "F"]

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.38, 0.62],
    shared_yaxes=True,
    horizontal_spacing=0.04
)

# ── Pannello sinistro: totale atleti per anno ─────────────────────────────
total_vals = total_year.set_index("Year").reindex(years)

fig.add_trace(
    go.Bar(
        y=years,
        x=total_vals["total"],
        orientation="h",
        marker_color="#6272a4",
        showlegend=False,
        name="Totale",
        hovertemplate="Anno %{y}: %{x} atleti<extra></extra>"
    ),
    row=1, col=1
)

# ── Pannello destro: 100% stacked M/F ────────────────────────────────────
for sex in sex_order:
    sub = part[part["Sex"] == sex].set_index("Year")
    total_dict = total_year.set_index("Year")["total"]
    pct = [(sub.loc[yr, "count"] / total_dict[yr] * 100
            if yr in sub.index else 0)
           for yr in years]

    fig.add_trace(
        go.Bar(
            y=years,
            x=pct,
            orientation="h",
            name="Uomini" if sex == "M" else "Donne",
            marker_color=color_sex[sex],
            hovertemplate=f"{'Uomini' if sex == 'M' else 'Donne'}: %{{x:.1f}}%<extra></extra>"
        ),
        row=1, col=2
    )

# ── Layout ────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(text="Partecipazione Olimpica Per Anno e Sesso", font_size=26, x=0),
    barmode="stack",
    template="plotly_white",
    height=680,
    legend=dict(title="Sesso", traceorder="normal",
                x=1.01, y=0.5, xanchor="left"),
    margin=dict(l=20, r=120, t=80, b=40),
    annotations=[
        dict(text="Atleti totali", x=0.17, y=1.04,
             xref="paper", yref="paper", showarrow=False, font_size=13),
        dict(text="Quota per sesso [%]", x=0.72, y=1.04,
             xref="paper", yref="paper", showarrow=False, font_size=13),
    ]
)

fig.update_xaxes(title_text="N° atleti", row=1, col=1)
fig.update_xaxes(title_text="Quota [%]", range=[0, 100], row=1, col=2)
fig.update_yaxes(title_text="Anno", tickmode="linear", dtick=4, row=1, col=1)

fig.show()